## Section 0 — Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U transformers datasets peft sentence-transformers faiss-cpu wandb scikit-learn lightgbm torchao

In [ ]:
import os, gc, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import wandb

WANDB_PROJECT = "24f2003563-t22026"

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

In [ ]:
def build_dataset(df, has_answer=True):
    rows = []
    for _, row in df.iterrows():
        for option in ['A', 'B', 'C', 'D', 'E']:
            text = f"Question: {row['prompt']} Candidate Answer: {row[option]}"
            item = {
                "id": row['id'],
                "prompt": row['prompt'],
                "option_text": row[option],
                "text": text,
                "option": option
            }
            if has_answer:
                item["label"] = 1 if option == row['answer'] else 0
            rows.append(item)
    return pd.DataFrame(rows)

train_processed = build_dataset(train, has_answer=True)
test_processed  = build_dataset(test,  has_answer=False)

print(train_processed.shape, test_processed.shape)
train_processed.head(10)

In [ ]:
# Train / Val split
train_ids, val_ids = train_test_split(train['id'].unique(), test_size=0.15, random_state=42)

train_df = train_processed[train_processed['id'].isin(train_ids)].reset_index(drop=True)
val_df   = train_processed[train_processed['id'].isin(val_ids)].reset_index(drop=True)

train_q = train[train['id'].isin(train_ids)].reset_index(drop=True)
val_q   = train[train['id'].isin(val_ids)].reset_index(drop=True)

print(f"Train questions: {len(train_ids)} | Val questions: {len(val_ids)}")


In [ ]:
# MAP@3 metric
def map_at_3(true_answers, pred_lists):
    """
    true_answers: list of correct option letters, e.g. ['A','C',...]
    pred_lists:   list of lists of top-3 predicted letters, e.g. [['A','B','C'], ...]
    """
    scores = []
    for true, preds in zip(true_answers, pred_lists):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

def scores_to_submission(id_list, option_list, score_list, out_path=None):
    """Convert per-option scores into a ranked top-3 submission dataframe."""
    df = pd.DataFrame({"id": id_list, "option": option_list, "score": score_list})
    rows = []
    for qid, g in df.groupby("id"):
        ranked = g.sort_values("score", ascending=False)
        top3 = ranked["option"].values[:3].tolist()
        rows.append({"ID": qid, "Prediction": " ".join(top3)})
    sub = pd.DataFrame(rows)
    if out_path:
        sub.to_csv(out_path, index=False)
    return sub


## W&B Login

In [ ]:
!wandb online

from kaggle_secrets import UserSecretsClient
import wandb

api_key = UserSecretsClient().get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb.init(
    entity="24f2003563-iit-madras",
    project="24f2003563-t22026"
)

## TF-IDF / Word2Vec Baseline + Cosine Similarity


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df["text_clean"] = train_df["text"].apply(clean_text)
val_df["text_clean"]   = val_df["text"].apply(clean_text)
test_processed["text_clean"] = test_processed["text"].apply(clean_text)

In [ ]:
# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df["text_clean"])
X_val_tfidf   = tfidf.transform(val_df["text_clean"])
X_test_tfidf  = tfidf.transform(test_processed["text_clean"])

prompt_vec_train = tfidf.transform(train_df["prompt"].apply(clean_text))
option_vec_train = tfidf.transform(train_df["option_text"].apply(clean_text))
cos_sim_train = np.array([
    cosine_similarity(prompt_vec_train[i], option_vec_train[i])[0, 0]
    for i in range(prompt_vec_train.shape[0])
])

prompt_vec_val = tfidf.transform(val_df["prompt"].apply(clean_text))
option_vec_val = tfidf.transform(val_df["option_text"].apply(clean_text))
cos_sim_val = np.array([
    cosine_similarity(prompt_vec_val[i], option_vec_val[i])[0, 0]
    for i in range(prompt_vec_val.shape[0])
])

prompt_vec_test = tfidf.transform(test_processed["prompt"].apply(clean_text))
option_vec_test = tfidf.transform(test_processed["option_text"].apply(clean_text))
cos_sim_test = np.array([
    cosine_similarity(prompt_vec_test[i], option_vec_test[i])[0, 0]
    for i in range(prompt_vec_test.shape[0])
])

print("Cosine sim feature ready. Example values:", cos_sim_train[:5])


In [ ]:
from scipy.sparse import hstack, csr_matrix

X_train_full = hstack([X_train_tfidf, csr_matrix(cos_sim_train).T])
X_val_full   = hstack([X_val_tfidf,   csr_matrix(cos_sim_val).T])
X_test_full  = hstack([X_test_tfidf,  csr_matrix(cos_sim_test).T])

y_train = train_df["label"].values
y_val   = val_df["label"].values

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_full, y_train)

val_probs_tfidf = logreg.predict_proba(X_val_full)[:, 1]
val_preds_tfidf = (val_probs_tfidf > 0.5).astype(int)

acc_tfidf = accuracy_score(y_val, val_preds_tfidf)
f1_tfidf  = f1_score(y_val, val_preds_tfidf)
print(f"TF-IDF + LogReg — Val Accuracy: {acc_tfidf:.4f} | F1: {f1_tfidf:.4f}")


In [ ]:
# Compute MAP@3 for TF-IDF model
val_sub_tfidf = scores_to_submission(val_df["id"].values, val_df["option"].values, val_probs_tfidf)
val_answers = val_q.set_index("id").loc[val_sub_tfidf["ID"], "answer"].values
pred_lists_tfidf = val_sub_tfidf["Prediction"].str.split().tolist()

map3_tfidf = map_at_3(val_answers, pred_lists_tfidf)
print(f"TF-IDF + LogReg — Val MAP@3: {map3_tfidf:.4f}")


In [ ]:
# Log this run to W&B
run = wandb.init(project=WANDB_PROJECT, name="m1-tfidf-logreg", config={
    "model": "TF-IDF + LogisticRegression",
    "max_features": 20000,
    "ngram_range": "(1,2)"
})
wandb.log({"val_accuracy": acc_tfidf, "val_f1": f1_tfidf, "val_map3": map3_tfidf})
run.finish()


## Transformer Embeddings + Zero-Shot Classification

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))

def zero_shot_score(df):
    prompts = df["prompt"].tolist()
    options = df["option_text"].tolist()
    prompt_emb = embed_model.encode(prompts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    option_emb = embed_model.encode(options, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    # cosine similarity per row
    sims = np.sum(prompt_emb * option_emb, axis=1) / (
        np.linalg.norm(prompt_emb, axis=1) * np.linalg.norm(option_emb, axis=1) + 1e-8
    )
    return sims

val_scores_zeroshot = zero_shot_score(val_df)


In [ ]:
val_sub_zeroshot = scores_to_submission(val_df["id"].values, val_df["option"].values, val_scores_zeroshot)
pred_lists_zeroshot = val_sub_zeroshot["Prediction"].str.split().tolist()
map3_zeroshot = map_at_3(val_answers, pred_lists_zeroshot)
print(f"Zero-shot MiniLM embeddings — Val MAP@3: {map3_zeroshot:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m2-zeroshot-embeddings", config={
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "method": "zero-shot cosine similarity"
})
wandb.log({"val_map3": map3_zeroshot})
run.finish()


## RAG Pipeline (Retrieval Augmentation)

In [ ]:
import faiss

corpus_texts = train["prompt"].tolist()
corpus_emb = embed_model.encode(corpus_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
corpus_emb = corpus_emb / (np.linalg.norm(corpus_emb, axis=1, keepdims=True) + 1e-8)

index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb.astype("float32"))
print("FAISS index built with", index.ntotal, "vectors")


In [ ]:
def retrieve_context(query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-8)
    scores, idxs = index.search(q_emb.astype("float32"), k)
    return [corpus_texts[i] for i in idxs[0]]

example_q = val_q.iloc[0]["prompt"]
print("Query:", example_q)
print("Retrieved context:", retrieve_context(example_q, k=2))


In [ ]:
def build_rag_text(row, k=2):
    context = retrieve_context(row["prompt"], k=k)
    context_str = " | ".join(context)
    return f"Context: {context_str} Question: {row['prompt']} Candidate Answer: {row['option_text']}"

val_df_rag = val_df.copy()
val_df_rag["rag_text"] = val_df_rag.apply(lambda r: build_rag_text(r, k=2), axis=1)
val_df_rag[["text", "rag_text"]].head(3)


In [ ]:
# Score RAG-augmented text
rag_prompt_emb = embed_model.encode(val_df_rag["rag_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_option_emb = embed_model.encode(val_df_rag["option_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_sims = np.sum(rag_prompt_emb * rag_option_emb, axis=1) / (
    np.linalg.norm(rag_prompt_emb, axis=1) * np.linalg.norm(rag_option_emb, axis=1) + 1e-8
)

val_sub_rag = scores_to_submission(val_df_rag["id"].values, val_df_rag["option"].values, rag_sims)
pred_lists_rag = val_sub_rag["Prediction"].str.split().tolist()
map3_rag = map_at_3(val_answers, pred_lists_rag)
print(f"RAG-augmented zero-shot — Val MAP@3: {map3_rag:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m3-rag-pipeline", config={"k_retrieved": 2})
wandb.log({"val_map3": map3_rag})
run.finish()


## LoRA Fine-Tuning (DeBERTa-v3)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType

model_name = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=384)


In [ ]:
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

class MCQDataset(TorchDataset):
    def __init__(self, hf_dataset):
        data = hf_dataset.to_dict()
        self.input_ids      = torch.tensor(data["input_ids"])
        self.attention_mask = torch.tensor(data["attention_mask"])
        self.labels         = torch.tensor(data["labels"]) if "labels" in data else None
        self.token_type_ids = torch.tensor(data["token_type_ids"]) if "token_type_ids" in data else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx]}
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

hf_train = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
hf_train = hf_train.rename_column("label", "labels")
hf_val   = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True)
hf_val   = hf_val.rename_column("label", "labels")

torch_train = MCQDataset(hf_train)
torch_val   = MCQDataset(hf_val)
print(f"Train rows: {len(torch_train)} | Val rows: {len(torch_val)}")


In [ ]:
# Class-weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        weight = torch.tensor([1.0, 4.0]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    pos_rate = preds.mean()   # sanity check: should land near 0.2
    return {"accuracy": acc, "f1": f1, "pred_positive_rate": pos_rate}


In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.2,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],  # DeBERTa-v3 attention projections
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()   # confirms only a tiny % of params are trainable


In [ ]:
steps_per_epoch = max(1, len(torch_train) // (8 * 4))
total_steps = steps_per_epoch * 6
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir="lora_outputs_v2",
    learning_rate=2e-4,           # lowered from 2e-4 — gentler updates
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=6,           # up from 4
    weight_decay=0.01,            # up from 0.01 — more regularization
    max_grad_norm=1.0,
    warmup_steps=warmup_steps,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="m4-lora-v2-regularized",
)

trainer = WeightedTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=torch_train,
    eval_dataset=torch_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


In [ ]:
eval_metrics_lora = trainer.evaluate()
print(eval_metrics_lora)

In [ ]:
# Get LoRA model
lora_model.eval()
val_loader_simple = torch.utils.data.DataLoader(torch_val, batch_size=32)

all_probs_lora = []
with torch.no_grad():
    for batch in val_loader_simple:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        out = lora_model(**batch)
        probs = F.softmax(out.logits.float(), dim=-1)[:, 1]
        all_probs_lora.extend(probs.cpu().numpy())

val_sub_lora = scores_to_submission(val_df["id"].values, val_df["option"].values, np.array(all_probs_lora))
pred_lists_lora = val_sub_lora["Prediction"].str.split().tolist()
map3_lora = map_at_3(val_answers, pred_lists_lora)
print(f"LoRA DeBERTa-v3 — Val MAP@3: {map3_lora:.4f}")

wandb.init(project=WANDB_PROJECT, name="m4-lora-final-map3", config={"model": "deberta-v3-large+LoRA"})
wandb.log({"val_map3": map3_lora, **eval_metrics_lora})
wandb.finish()


In [ ]:
# Build RAG-augmented text
test_processed_rag = test_processed.copy()
test_processed_rag["rag_text"] = test_processed_rag.apply(lambda r: build_rag_text(r, k=2), axis=1)

# ── Score RAG-augmented text using the same zero-shot embedding approach ──
rag_prompt_emb_test = embed_model.encode(test_processed_rag["rag_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_option_emb_test = embed_model.encode(test_processed_rag["option_text"].tolist(), batch_size=32, show_progress_bar=True)

test_rag_scores = np.sum(rag_prompt_emb_test * rag_option_emb_test, axis=1) / (
    np.linalg.norm(rag_prompt_emb_test, axis=1) * np.linalg.norm(rag_option_emb_test, axis=1) + 1e-8
)

print("test_rag_scores ready:", test_rag_scores.shape)

## Milestone 5: Ensembling


In [ ]:
def normalize(scores):
    scores = np.array(scores, dtype=float)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

ensemble_scores = (
    0.2 * normalize(val_probs_tfidf) +
    0.2 * normalize(val_scores_zeroshot) +
    0.6 * normalize(all_probs_lora)        # weight the strongest model highest
)

val_sub_ensemble = scores_to_submission(val_df["id"].values, val_df["option"].values, ensemble_scores)
pred_lists_ensemble = val_sub_ensemble["Prediction"].str.split().tolist()
map3_ensemble = map_at_3(val_answers, pred_lists_ensemble)
print(f"Ensemble (TFIDF + ZeroShot + LoRA) — Val MAP@3: {map3_ensemble:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m5-ensemble", config={
    "weights": {"tfidf": 0.2, "zeroshot": 0.2, "lora": 0.6}
})
wandb.log({"val_map3": map3_ensemble})
run.finish()


In [ ]:
summary = pd.DataFrame([
    {"model": "TF-IDF + LogisticRegression", "val_map3": map3_tfidf, "val_f1": f1_tfidf, "val_acc": acc_tfidf},
    {"model": "Zero-shot MiniLM embeddings", "val_map3": map3_zeroshot},
    {"model": "RAG-augmented embeddings",    "val_map3": map3_rag},
    {"model": "LoRA fine-tuned DeBERTa-v3",  "val_map3": map3_lora, **eval_metrics_lora},
    {"model": "Ensemble (all 3 main models)","val_map3": map3_ensemble},
])
summary.to_csv("model_comparison.csv", index=False)
summary

## Final Submission


In [ ]:
test_processed["text_clean"] = test_processed["text"].apply(clean_text)

# TF-IDF scores on test
test_probs_tfidf = logreg.predict_proba(X_test_full)[:, 1]

# Zero-shot embedding scores on test
test_scores_zeroshot = zero_shot_score(test_processed)

# LoRA model scores on test
hf_test = Dataset.from_pandas(test_processed[["text"]]).map(tokenize, batched=True)
torch_test = MCQDataset(hf_test)
test_loader = torch.utils.data.DataLoader(torch_test, batch_size=32)

lora_model.eval()
test_probs_lora = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = lora_model(**batch)
        probs = F.softmax(out.logits.float(), dim=-1)[:, 1]
        test_probs_lora.extend(probs.cpu().numpy())
test_probs_lora = np.array(test_probs_lora)


In [ ]:
best_score = -1
best_weights = None

for w_lora in [0.5, 0.6, 0.65, 0.7, 0.75]:
    remaining = 1 - w_lora
    w_tfidf = remaining * 0.3
    w_zero = remaining * 0.2
    w_rag = remaining * 0.5

    scores = (
        w_tfidf * normalize(val_probs_tfidf) +
        w_zero  * normalize(val_scores_zeroshot) +
        w_rag   * normalize(rag_sims) +
        w_lora  * normalize(all_probs_lora)
    )

    sub = scores_to_submission(val_df["id"].values, val_df["option"].values, scores)
    preds = sub["Prediction"].str.split().tolist()
    m3 = map_at_3(val_answers, preds)

    val_probs_lora = np.array(all_probs_lora)
    print(f"w_lora={w_lora} | tfidf={w_tfidf:.3f} zero={w_zero:.3f} rag={w_rag:.3f} | val MAP@3={m3:.4f}")

    if m3 > best_score:
        best_score = m3
        best_weights = (w_tfidf, w_zero, w_rag, w_lora)

print("\nBEST:", best_weights, "-> val MAP@3:", best_score)

In [ ]:
test_ensemble_scores_v3 = (
    0.10 * normalize(test_probs_tfidf) +
    0.05 * normalize(test_scores_zeroshot) +
    0.20 * normalize(test_rag_scores) +
    0.65 * normalize(test_probs_lora)
)

final_submission_v3 = scores_to_submission(
    test_processed["id"].values,
    test_processed["option"].values,
    test_ensemble_scores_v3,
    out_path="submission.csv"
)